# 01 — The Serra do Mar 2011 flash-flood event

[Open in Colab](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/01_event_serra_do_mar_2011.ipynb)

**What we're investigating.** On the night of 11–12 January 2011 the Serra do
Mar mountains of Rio de Janeiro state received hundreds of mm of rain in a
few hours. Landslides killed more than 900 people across Nova Friburgo,
Teresópolis and neighbouring towns — the worst natural disaster in Brazilian
history. This notebook asks **where did that rain come from?** by reading
AguaTrack's backward-tracking output and rendering daily moisture-source maps
for the 18-day window 2011-01-05 to 2011-01-22.

**What you'll get.** Three animated GIFs (54 frames total: 18 days x 3
views), plus one standalone time-series figure showing the basin-mean tagged
precipitation through the whole month of January 2011. The three views are
complementary and answer slightly different versions of the question:

- **abs**: absolute source strength in mm/day water-equivalent. Best for
  "how much rain came from this pixel?"
- **pdf**: the same field normalised so each daily map integrates to 1 over
  space. Best for "what fraction of *today's* moisture came from here?"
- **cdf**: per-day spatial rank-cumulative percentile. Best for drawing
  "the smallest set of pixels covering 90% of the source" contours.

**Compute footprint.** ~50 tag cells x 18 days x the full South America
grid -> a few hundred MB in memory. Comfortable on Colab Pro and tight on
the free tier. The 54 figures take 2–3 minutes to render.

**How to cite.** TODO: paper in submission.

## Step 1 — Configuration

Everything you might want to edit lives in this single cell:

- **Receptor box** — Serra do Mar by default. A rectangular box matches
  the ~500 km synoptic scale of the SACZ event well; the HYBAS level-2
  basin that contains Serra do Mar is ~40x larger and would average in
  unrelated rainfall from north-east Brazil.
- **Analysis window** — 18 days centred on the night of 11–12 Jan 2011
  (`EVENT_ONSET`). The full month is used for the time-series figure.

In [ ]:
HF_REVISION = "main"
AGUATRACK_DAILY_2011 = (
    "hf://datasets/AguaTrack/AguaTrack-ARCO-SA"
    "/AguaTrack_ARCO_SA_daily/2011.zarr"
)

# Serra do Mar bounding box — covers Nova Friburgo / Teresopolis / Rio metro.
BOX_LAT_S, BOX_LAT_N = -23.5, -21.5
BOX_LON_W, BOX_LON_E = -44.0, -41.0

# 18-day analysis window centred on the event onset (the GIF window).
DATE_START = "2011-01-05"
DATE_END = "2011-01-22"
# Full month bracket for the time-series figure at the end.
JAN_START, JAN_END = "2011-01-01", "2011-01-31"
EVENT_ONSET = "2011-01-11"  # night of 11->12 Jan: Nova Friburgo landslides begin

## Step 2 — Install dependencies (Colab only)

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask pillow",
    )

## Step 3 — Imports, plotting style, helper functions

We pull in cartopy for map projections, cmcrameri for the perceptually
uniform `batlowW` colour map (better than the default `viridis` for
physical fields), and matplotlib for the time-series figure.

The four helper functions below are short translations of the formulas
we apply to the daily source field. The maths is documented in the
docstrings.

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import xarray as xr

# Project-wide rcParams. Larger fonts than matplotlib's default since the
# figures end up in a paper / Colab where the screenshot is small.
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "figure.titlesize": 18,
})


def cos_lat_weights(lats_deg: np.ndarray) -> xr.DataArray:
    """cos(lat) weights along the tagging_mask axis.

    At 0.25 degree resolution every cell has the same lat/lon extent in
    degrees, so the cos(lat) factor is a stand-in for cell area. We use
    these weights when averaging across tag cells of different latitudes.
    """
    return xr.DataArray(np.cos(np.deg2rad(lats_deg)), dims="tagging_mask")


def precip_weighted_source(
    e_region: xr.DataArray,
    tp_window: xr.DataArray,
    tag_area_w: xr.DataArray,
) -> xr.DataArray:
    """Precipitation-and-area-weighted moisture-source field per day.

    Why weight by both precip and area? Two reasons:
    1) cos(lat) area weighting prevents the smallest (highest-latitude)
       cells from dominating the basin mean.
    2) precipitation weighting answers the right question. We want
       "where did *today's* tagged rain come from?", not "where did the
       average tag's evaporation come from?". On a wet day, dry tags
       contribute a near-zero weight and quietly drop out. On a fully
       dry day, all weights are zero — we set the result to NaN there
       (there is nothing to attribute).
    """
    rain_weights = tp_window * tag_area_w           # shape (time, tag)
    rain_total = rain_weights.sum("tagging_mask")    # shape (time,)
    safe_denom = rain_total.where(rain_total > 0)    # NaN on dry days
    return (e_region * rain_weights).sum("tagging_mask") / safe_denom


def source_pdf(field: xr.DataArray, lat_weights_2d: xr.DataArray) -> xr.DataArray:
    """Normalise a daily source field so each day integrates to 1 over space.

    The integration is area-weighted (cos(lat)) so the resulting "PDF"
    really is dimensionless and sums to 1 when integrated over the
    spherical surface. Useful for comparing the *shape* of the source
    region between days that differ wildly in absolute strength.
    """
    integral = (field * lat_weights_2d).sum(("latitude", "longitude"))
    safe = integral.where(integral > 0)
    return field / safe


def source_cdf(field: xr.DataArray) -> xr.DataArray:
    """Per-day spatial rank-cumulative percentile.

    For each day, sort all positive pixels from largest to smallest,
    cumulatively sum, and divide by the total. The pixel with the
    largest contribution gets a value near 0%; the smallest gets a value
    near 100%. The resulting field is great for drawing contours like
    "the smallest set of pixels covering 90% of today's source".
    """
    out = xr.full_like(field, np.nan, dtype=float)
    for t in range(field.sizes["time"]):
        vals = field.isel(time=t).values
        flat = vals.ravel()
        # Drop NaNs (dry days) and zero pixels (no evaporation traced)
        mask = np.isfinite(flat) & (flat > 0)
        if not mask.any():
            continue
        valid = flat[mask]
        # Sort descending, cumulative sum, normalise to [0, 100]
        order_desc = np.argsort(valid)[::-1]
        cum_pct = 100.0 * np.cumsum(valid[order_desc]) / valid.sum()
        # Scatter the cumulative percentages back to their pixel positions
        flat_cdf = np.full_like(flat, np.nan, dtype=float)
        valid_positions = np.where(mask)[0]
        flat_cdf[valid_positions[order_desc]] = cum_pct
        out.values[t] = flat_cdf.reshape(vals.shape)
    return out

## Step 4 — Open the 2011 zarr and find the tag cells inside the box

`box_tag_idx` is the integer index array we will pass to `isel` later.

In [ ]:
# Open the 2011 daily zarr — lazy, no data transferred yet.
ds = xr.open_zarr(AGUATRACK_DAILY_2011, storage_options={"revision": HF_REVISION})

# Build the tag-cell mask in pure xarray (lazy if the coords are dask-backed,
# eager if numpy-backed — either way xarray triggers compute when we call
# .values on the small 1-D mask). No explicit .compute() / .load() needed.
in_box = (
    (ds.tag_lat >= BOX_LAT_S) & (ds.tag_lat <= BOX_LAT_N)
    & (ds.tag_lon >= BOX_LON_W) & (ds.tag_lon <= BOX_LON_E)
)
box_tag_idx = np.flatnonzero(in_box.values)
print(f"tags inside box: {len(box_tag_idx)}")
# Pull the per-tag lat/lon for the box subset only (~50 floats).
box_lat = ds.tag_lat.isel(tagging_mask=box_tag_idx)
box_lon = ds.tag_lon.isel(tagging_mask=box_tag_idx)
print(
    f"  lat range {float(box_lat.min()):.2f}..{float(box_lat.max()):.2f}, "
    f"lon range {float(box_lon.min()):.2f}..{float(box_lon.max()):.2f}"
)

## Step 5 — Load the bytes

Two memory considerations matter here:

1. **Pre-select before load.** The native zarr chunking is
   `(365, 10, 301, 261)` ≈ 1.15 GB per chunk; loading without an
   `isel(tagging_mask=...)` would touch every chunk in the year.
2. **Re-chunk to one tag at a time.** Even with our ~50-tag selection,
   dask would otherwise materialise 5 native chunks (~5.7 GB
   decompressed) before pruning to the 18-day slice. Re-chunking to
   `tagging_mask=1` lets dask stream the read tag-by-tag — peak RAM
   drops to a few hundred MB, comfortable on Colab free tier.

After the `.load()` boundary, `e_region` and `tp_region` are eager
in-memory `xarray.DataArray`s. The rest of the notebook is pure xarray
(no further `.load()` / `.compute()` calls).

In [ ]:
from dask.diagnostics import ProgressBar  # standard Pangeo progress bar

days = pd.date_range(DATE_START, DATE_END).strftime("%Y-%m-%d").tolist()
jan_days = pd.date_range(JAN_START, JAN_END).strftime("%Y-%m-%d").tolist()

# Build a lazy view first, then re-chunk + load. The re-chunk is what
# keeps Colab free-tier memory comfortable.
e_lazy = (
    ds.e_track.isel(tagging_mask=box_tag_idx).sel(time=days)
    .chunk({"tagging_mask": 1, "time": -1, "latitude": -1, "longitude": -1})
)
tp_lazy = ds.tagged_precip.isel(tagging_mask=box_tag_idx).sel(time=jan_days)

with ProgressBar():
    e_region = e_lazy.load()
    tp_region = tp_lazy.load()
print(f"e_region: shape={e_region.shape}  size={e_region.nbytes/1e9:.2f} GB in RAM")

## Step 6 — Build the three source views

We start with `e_abs`, the precipitation- and area-weighted source field
(same units as `e_track`, mm/day). Then we derive the dimensionless `e_pdf`
by per-day area-normalisation, and the percentile field `e_cdf` by per-day
rank ordering. All three share the same `(time, lat, lon)` shape.

We also pre-compute shared colour-bar limits (`VMAX_ABS`, `VMAX_PDF`) using
the 99th percentile of non-zero pixels across the whole 18-day window.
Using the same scale across all 18 frames makes the GIF intensity directly
comparable from day to day.

In [ ]:
# cos(lat) weights along the box's tag axis.
tag_area_w = cos_lat_weights(box_lat.values)

# 2-D cos(lat) weights for area-normalisation of the source maps. We rely
# on xarray automatic broadcasting: a 1-D DataArray on `latitude` times a
# 1-D DataArray of ones on `longitude` gives the right (lat, lon) field.
grid_area_w = (
    np.cos(np.deg2rad(ds.latitude))
    * xr.ones_like(ds.longitude, dtype=float)
)

# tp_window is the rain we will use as a weight in `precip_weighted_source`.
# We slice tp_region (full Jan) down to the 18 event days.
tp_window = tp_region.sel(time=days)
e_abs = precip_weighted_source(e_region, tp_window, tag_area_w)   # mm/day
e_pdf = source_pdf(e_abs, grid_area_w)                            # dimensionless
e_cdf = source_cdf(e_abs)                                         # 0-100 %

# Basin-mean tagged precip for the time-series figure (full month context).
# Pure xarray: `weighted` handles the area-weighting correctly, including
# NaN propagation.
tp_basin_mean = tp_region.weighted(tag_area_w).mean("tagging_mask")

# Shared colour-bar limits — 99th percentile of non-zero pixels. xarray's
# `quantile` ignores NaN by default; we mask zeros first with `.where`.
e_abs_pos = e_abs.where(e_abs > 0)
e_pdf_pos = e_pdf.where(e_pdf > 0)
VMAX_ABS = float(e_abs_pos.quantile(0.99))
VMAX_PDF = float(e_pdf_pos.quantile(0.99))
TP_YMAX = float(tp_basin_mean.max()) * 1.1

# CDF contour levels — the "top X% of source" boundaries we draw on the
# CDF maps.
CDF_LEVELS = [10, 30, 50, 70, 90]

print(f"vmax abs (99th pctile of non-zero pixels): {VMAX_ABS:.4f} kg m-2 day-1")
print(f"vmax pdf (99th pctile of non-zero pixels): {VMAX_PDF:.3e}  (dimensionless)")

## Step 7 — Plot helpers

`decorate_map` adds the standard map decoration (coastlines, the red box
around the receptor region, gridlines). `map_figure` is a small factory
that returns a square 4-inch figure with a PlateCarree projection — the
tracking domain is roughly square (65 deg longitude x 75 deg latitude),
so a square figure fills the canvas without whitespace.

In [ ]:
def decorate_map(ax):
    """Apply the standard map decoration to a cartopy axis."""
    # 50m coastlines — finer than 110m, coarse enough for a continental view.
    ax.coastlines(resolution="50m", color="black", linewidth=1)

    # Highlight the receptor box in red so readers see *where* the rain
    # is being attributed to.
    ax.add_patch(mpatches.Rectangle(
        (BOX_LON_W, BOX_LAT_S),
        BOX_LON_E - BOX_LON_W, BOX_LAT_N - BOX_LAT_S,
        linewidth=2, edgecolor="red", facecolor="none",
        transform=ccrs.PlateCarree(), zorder=5,
    ))

    # Lock the visible extent to the dataset's tracking domain.
    ax.set_xlim(ds.longitude.min(), ds.longitude.max())
    ax.set_ylim(ds.latitude.min(), ds.latitude.max())

    # 10-degree gridlines, no grid lines drawn (linewidth=0) — we only
    # want the tick labels.
    gl = ax.gridlines(draw_labels=True, linewidth=0.0)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 10))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 10))
    gl.xlabel_style = {"size": 12}
    gl.ylabel_style = {"size": 12}


def map_figure():
    """Square 4-inch figure with a PlateCarree projection."""
    fig = plt.figure(figsize=(4, 4), constrained_layout=True)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    return fig, ax

## Step 8 — Render the per-day frames

18 days x 3 views = 54 PNG files. We mask out zero pixels with
`da.where(da > 0)` to keep the colour bar clean (zeros would otherwise
saturate the dark end of the perceptual colour map).

In [ ]:
OUT_ABS = Path("outputs/box_abs"); OUT_ABS.mkdir(parents=True, exist_ok=True)
OUT_PDF = Path("outputs/box_pdf"); OUT_PDF.mkdir(parents=True, exist_ok=True)
OUT_CDF = Path("outputs/box_cdf"); OUT_CDF.mkdir(parents=True, exist_ok=True)


def draw_abs(day, out_png):
    """Absolute source strength (mm/day water-equivalent)."""
    fig, ax = map_figure()
    da = e_abs.sel(time=day)
    mesh = ax.pcolormesh(
        da.longitude, da.latitude, da.where(da > 0),
        cmap=cmc.batlowW_r, vmin=0, vmax=VMAX_ABS,
        transform=ccrs.PlateCarree(), shading="auto",
    )
    decorate_map(ax)
    ax.set_title(day)
    cb = fig.colorbar(mesh, ax=ax, shrink=0.8, pad=0.02)
    cb.set_label("Tracked evaporation [kg m^-2 day^-1]")
    cb.ax.tick_params(labelsize=12)
    fig.savefig(out_png, dpi=150, pad_inches=0.1, facecolor="white")
    plt.close(fig)


def draw_pdf(day, out_png):
    """Area-normalised source PDF (dimensionless, integrates to 1)."""
    fig, ax = map_figure()
    da = e_pdf.sel(time=day)
    mesh = ax.pcolormesh(
        da.longitude, da.latitude, da.where(da > 0),
        cmap=cmc.batlowW_r, vmin=0, vmax=VMAX_PDF,
        transform=ccrs.PlateCarree(), shading="auto",
    )
    decorate_map(ax)
    ax.set_title(day)
    cb = fig.colorbar(mesh, ax=ax, shrink=0.8, pad=0.02)
    cb.set_label("Source PDF (area-weighted, integrates to 1)")
    cb.ax.tick_params(labelsize=12)
    fig.savefig(out_png, dpi=150, pad_inches=0.1, facecolor="white")
    plt.close(fig)


def draw_cdf(day, out_png):
    """Per-day spatial rank-cumulative percentile."""
    fig, ax = map_figure()
    da = e_cdf.sel(time=day)
    # contourf for the filled bands, contour for the percentile lines.
    cf = ax.contourf(
        da.longitude, da.latitude, da,
        levels=CDF_LEVELS + [100.0],
        cmap=cmc.batlowW, extend="neither",
        transform=ccrs.PlateCarree(),
    )
    ax.contour(
        da.longitude, da.latitude, da,
        levels=CDF_LEVELS, colors="black", linewidths=0.5,
        transform=ccrs.PlateCarree(),
    )
    decorate_map(ax)
    ax.set_title(f"{day}\ncumulative source percentile (per-day)", fontsize=12)
    cb = fig.colorbar(cf, ax=ax, shrink=0.8, pad=0.02, ticks=CDF_LEVELS + [100])
    cb.set_label("Cumulative share of tracked evaporation [%]")
    cb.ax.tick_params(labelsize=12)
    fig.savefig(out_png, dpi=150, pad_inches=0.1, facecolor="white")
    plt.close(fig)


# Render all 54 frames.
for d in days:
    draw_abs(d, OUT_ABS / f"{d}.png")
    draw_pdf(d, OUT_PDF / f"{d}.png")
    draw_cdf(d, OUT_CDF / f"{d}.png")
print(f"rendered {len(days)} frames x 3 views in outputs/box_{{abs,pdf,cdf}}/")

## Step 9 — Assemble the GIFs

PIL is plenty fast for 18-frame GIFs and is pre-installed on Colab. One
GIF per view, one second per frame.

In [ ]:
from PIL import Image

OUT_ROOT = Path("outputs")


def assemble_gif(frame_dir: Path, out_path: Path, fps: float = 1.0):
    """Stitch a directory of PNG frames into one looping GIF."""
    frames = sorted(frame_dir.glob("*.png"))
    if not frames:
        print(f"  skip {out_path.name} — no frames")
        return
    images = [Image.open(p).convert("RGB") for p in frames]
    images[0].save(
        out_path, save_all=True, append_images=images[1:],
        duration=int(1000 / fps), loop=0,
    )
    print(f"  wrote {out_path}  ({len(frames)} frames)")


for view in ("abs", "pdf", "cdf"):
    assemble_gif(OUT_ROOT / f"box_{view}", OUT_ROOT / f"box_{view}.gif")

## Step 10 — Display the GIFs inline

In [ ]:
from IPython.display import Image as IPyImage, display

for view in ("abs", "pdf", "cdf"):
    p = OUT_ROOT / f"box_{view}.gif"
    if p.exists():
        print(view)
        display(IPyImage(filename=str(p)))

## Step 11 — Standalone monthly time-series of tagged precipitation

A context figure for the GIF: how did basin-mean tagged rainfall over the
Serra do Mar box evolve through the whole of January 2011? We use a
step + fill hybrid:

- the step line preserves the daily discretisation (the data is daily, not
  continuous);
- the fill conveys the temporal envelope of the event;
- the event window (11–22 Jan) is shaded more strongly than the
  precursor/recovery days;
- a dashed crimson line marks the event onset; the peak day is annotated.

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(16, 7))

times = tp_basin_mean.time.values
values = tp_basin_mean.values
onset = np.datetime64(EVENT_ONSET)
active_end = np.datetime64(DATE_END)
active_mask = (times >= onset) & (times <= active_end)

# Two passes of fill_between: a light wash over the whole month, plus a
# stronger wash over the active event window.
ax.fill_between(times, 0, values, step="mid",
                color="steelblue", alpha=0.25, linewidth=0)
ax.fill_between(times, 0, values, step="mid",
                where=active_mask,
                color="steelblue", alpha=0.7, linewidth=0)
# Step outline on top.
ax.step(times, values, where="mid",
        color="black", linewidth=1.2, zorder=3)

# Event-onset line + label.
ax.axvline(onset, color="crimson", linestyle="--", linewidth=2, zorder=5)
ax.annotate(
    f"event onset\n{EVENT_ONSET}",
    xy=(onset - np.timedelta64(2, 'D'), TP_YMAX * 0.7), xytext=(8, 0),
    textcoords="offset points",
    color="crimson", fontsize=13, ha="right", va="center", fontweight="bold",
)

# Peak-day annotation.
peak_idx = int(np.argmax(values))
peak_time = times[peak_idx]
peak_val = float(values[peak_idx])
ax.annotate(
    f"peak {peak_val:.1f}\n{str(peak_time)[:10]}",
    xy=(peak_time, peak_val), xytext=(0, 12),
    textcoords="offset points",
    ha="center", va="bottom", fontsize=12,
    arrowprops=dict(arrowstyle="-", color="black", lw=0.6),
)

# Cosmetic axis setup.
ax.set_ylim(0, TP_YMAX)
ax.set_ylabel("Box-mean tagged precipitation\n[kg m^-2 day^-1]")
ax.set_title("Tagged precipitation over the Serra do Mar box, January 2011")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
ax.xaxis.set_minor_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ts_path = Path("outputs/box_tagged_precip_jan2011.png")
fig.tight_layout()
fig.savefig(ts_path, dpi=200, pad_inches=0.1, facecolor="white")
plt.show()
print(f"saved {ts_path}")